In [1]:
# ---------------------------
# Config - edit these if needed
# ---------------------------
FILE_PATH = r"S:\MCA PRACTICAL\3rd sem\Minor Project\TruthLens\Data\Preprocessed\Preprocessed_hindi_data.csv"
TEXT_COL  = "clean_joined"   # <-- your main Hindi text column
LABEL_COL = "label"          # <-- correct label column
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
OUT_DIR = r"models\light\hindi_light_model"
MIN_TEXT_LEN = 20
# ---------------------------


In [2]:
import os, json, re, numpy as np, pandas as pd
from pathlib import Path
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
print("Step 1/9 — Loading data from CSV...")
p = Path(FILE_PATH)
if not p.exists():
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

def load_data(path, usecols):
    try:
        df = pd.read_csv(path, usecols=usecols, encoding="utf-8", on_bad_lines="skip", low_memory=True)
        return df
    except Exception:
        import csv, sys
        try: csv.field_size_limit(sys.maxsize)
        except Exception: pass
        chunks = []
        for chunk in pd.read_csv(path, usecols=usecols, engine="python", encoding="utf-8",
                                 on_bad_lines="skip", chunksize=50000, sep=",",
                                 quotechar=None, quoting=csv.QUOTE_NONE, escapechar="\\"):
            chunks.append(chunk)
        return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)

df = load_data(FILE_PATH, usecols=[TEXT_COL, LABEL_COL])
print(f"Raw rows loaded: {len(df)}")


# BASIC CLEANING

print("Step 2/9 — Cleaning and basic filtering...")
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)
before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print(f"Removed duplicates: {before - len(df)}  (remaining rows: {len(df)})")


#  LABEL NORMALIZATION

print("Step 3/9 — Normalizing labels...")
# ---------------- Normalizing labels (fixed) ----------------
print("Step 3/9 — Normalizing labels...")

def normalize_labels(ser):
    s = ser.copy()
    # if numeric already, assume 0/1 (but coerce safely)
    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
        return s.fillna(0).astype(int)

    s = s.astype(str).str.strip().str.lower()

    # explicit mapping: fake -> 0, real -> 1
    mapping = {
        "fake": 0, "false": 0, "फेक": 0, "झूठा": 0, "0": 0,
        "real": 1, "true": 1, "सही": 1, "genuine": 1, "1": 1
    }
    mapped = s.map(mapping)

    if mapped.isnull().any():
        # fallback: if text contains any indicator-of-real -> 1, else 0
        def fallback_label(x):
            # check common tokens that strongly indicate 'real'
            real_indicators = ["real", "true", "सही", "genuine", "fact", "reported", "confirmed"]
            fake_indicators = ["fake", "false", "फेक", "झूठ", "fabricated", "hoax"]

            x_lower = str(x).lower()
            if any(tok in x_lower for tok in real_indicators):
                return 1
            if any(tok in x_lower for tok in fake_indicators):
                return 0
            # default conservative: treat unknown as fake (0)
            return 0

        mapped = mapped.fillna(s.apply(fallback_label))

    # final safety: coerce to int and force values to 0/1
    mapped = mapped.astype(int).clip(lower=0, upper=1)
    return mapped

y = normalize_labels(df[LABEL_COL])
X = df[TEXT_COL].astype(str)

# Sanity checks
print("Label distribution (after normalization):", y.value_counts(dropna=False).to_dict())
# ensure only {0,1} present
unique_labels = sorted(y.unique().tolist())
if unique_labels != [0, 1] and unique_labels not in ([0], [1]):
    print("⚠️ Unexpected label set:", unique_labels)

# Show a few example rows where mapping was ambiguous (optional)
ambig_idx = y[y.isin([0,1]) == False].index.tolist()  # should be empty by now
if ambig_idx:
    print("Examples of ambiguous original labels (first 5):")
    display(df.loc[ambig_idx[:5], [TEXT_COL, LABEL_COL]])

print("Step 4/9 — Creating stratified splits...")
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_STATE)
tr_idx, val_idx = next(sss.split(X_train_all, y_train_all))
X_tr, y_tr = X_train_all.iloc[tr_idx].reset_index(drop=True), y_train_all.iloc[tr_idx].reset_index(drop=True)
X_val, y_val = X_train_all.iloc[val_idx].reset_index(drop=True), y_train_all.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_test)}")

Step 1/9 — Loading data from CSV...
Raw rows loaded: 20593
Step 2/9 — Cleaning and basic filtering...
Removed duplicates: 4600  (remaining rows: 14854)
Step 3/9 — Normalizing labels...
Step 3/9 — Normalizing labels...
Label distribution (after normalization): {1: 7470, 0: 7384}
Step 4/9 — Creating stratified splits...
Train: 9506, Val: 2377, Test: 2971


In [4]:
# Config for speed (set fast_train=True for quick run)
fast_train = True        # True -> subsample training set and reduce grid-search; False -> full training
SUBSAMPLE_SIZE = 4000    # if fast_train: number of training rows to keep (balanced)
CALIBRATE_CV = 2         # calibration folds (smaller when fast_train)

print("Step 5/9 — Building TF-IDF + LinearSVC pipeline...")

# Build a picklable text cleaner (no lambdas)
def text_cleaner_list(X):
    """Accepts iterable X of strings, returns cleaned list of strings."""
    import re
    url_pat = re.compile(r"http\S+|www\.\S+")
    handle_pat = re.compile(r"[@#]\w+")
    multi_ws = re.compile(r"\s+")
    cleaned = []
    for s in X:
        s = str(s).lower()
        s = url_pat.sub(" ", s)
        s = handle_pat.sub(" ", s)
        s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
        s = multi_ws.sub(" ", s).strip()
        cleaned.append(s)
    return cleaned

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC

cleaner = FunctionTransformer(text_cleaner_list, validate=False)

word_vect = TfidfVectorizer(
    analyzer='word', token_pattern=r'(?u)\b\w+\b',
    ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode',
    stop_words='english'   # keep english stopwords — if Hindi model, remove or adapt later
)
char_vect = TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=2, sublinear_tf=True)

features = FeatureUnion([('word', word_vect), ('char', char_vect)])
selector = SelectKBest(chi2, k=20000)   # reduce to top K features
clf = LinearSVC(random_state=RANDOM_STATE, C=1.0, max_iter=20000)

pipeline = Pipeline([
    ('clean', cleaner),
    ('feats', features),
    ('select', selector),
    ('clf', clf),
])

# ---- Optional fast training: subsample balanced set ----
if fast_train:
    print("Fast training: subsample training data for speed.")
    # create a balanced subsample of X_tr / y_tr with SUBSAMPLE_SIZE (approx half/half by label)
    from sklearn.utils import resample
    import numpy as np
    df_tr = pd.DataFrame({'text': X_tr, 'label': y_tr})
    n = min(len(df_tr), SUBSAMPLE_SIZE)
    # stratified approximate subsample:
    df_sub = df_tr.groupby('label', group_keys=False).apply(lambda g: g.sample(max(1, int(n * len(g)/len(df_tr))), random_state=RANDOM_STATE)).reset_index(drop=True)
    # if still too large/small, sample exactly n while preserving stratification as best:
    if len(df_sub) > n:
        df_sub = df_sub.sample(n, random_state=RANDOM_STATE)
    X_tr_sub = df_sub['text'].tolist()
    y_tr_sub = df_sub['label'].astype(int).tolist()
else:
    X_tr_sub, y_tr_sub = X_tr.tolist(), y_tr.tolist()

print("Step 6/9 — Training model (fast or thorough depending on config)...")

# Minimal grid if fast, larger if full
from sklearn.model_selection import StratifiedKFold, GridSearchCV
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

if fast_train:
    param_grid = {
        'feats__word__ngram_range': [(1,1)],    # limit choices
        'feats__word__min_df': [2],
        'feats__char__ngram_range': [(3,5)],
        'select__k': [10000],
        'clf__C': [1.0],
    }
    cv_folds = 2
else:
    param_grid = {
        'feats__word__ngram_range': [(1,1), (1,2)],
        'feats__word__min_df': [1, 2],
        'feats__char__ngram_range': [(3,5), (3,6)],
        'select__k': [10000, 20000],
        'clf__C': [0.5, 1.0],
    }
    cv_folds = 3

grid = GridSearchCV(pipeline, param_grid, scoring='f1', cv=skf, verbose=1, n_jobs=1)
grid.fit(X_tr_sub, y_tr_sub)
print("Best params:", grid.best_params_)
print("Best CV f1: {:.4f}".format(grid.best_score_))

best_pipeline = grid.best_estimator_

# ---- Calibration for probabilities ----
print("Step 7/9 — Calibrating classifier to obtain probabilities...")
from sklearn.calibration import CalibratedClassifierCV
calib = CalibratedClassifierCV(best_pipeline, cv=CALIBRATE_CV, method='sigmoid')
# For calibration, use the same training subset (fast) or full training split otherwise
calib.fit(X_tr_sub, y_tr_sub)
print("Calibration done.")

# ---- Tune threshold on validation set ----
print("Step 8/9 — Tuning decision threshold on validation set (maximize F1)...")
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix
val_prob = calib.predict_proba(X_val)[:, 1]
best_th, best_f1 = 0.5, -1.0
import numpy as np
for t in np.linspace(0.2, 0.8, 61):
    f1 = f1_score(y_val, (val_prob >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_th = f1, t
print(f"Chosen threshold on validation: {best_th:.3f}  (val F1={best_f1:.4f})")

# ---- Evaluate on test ----
print("Step 9/9 — Evaluating on test set and saving artifacts...")
test_prob = calib.predict_proba(X_test)[:, 1]
y_pred = (test_prob >= best_th).astype(int)

metrics = {
    "test_accuracy": float(accuracy_score(y_test, y_pred)),
    "test_f1": float(f1_score(y_test, y_pred)),
    "test_roc_auc": None,
    "threshold": float(best_th),
    "n_train_used": int(len(X_tr_sub)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
}

try:
    metrics["test_roc_auc"] = float(roc_auc_score(y_test, test_prob))
except Exception:
    metrics["test_roc_auc"] = None

print("Test Accuracy:", metrics["test_accuracy"])
print("Test F1:", metrics["test_f1"])
print("Test ROC-AUC:", metrics["test_roc_auc"])
print("Classification report:\n", classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# ---- Save model + metrics + info ----
import os, json
from joblib import dump
os.makedirs(OUT_DIR, exist_ok=True)
model_obj = {"pipeline": calib, "threshold": float(best_th)}
model_path = os.path.join(OUT_DIR, "model.joblib")
dump(model_obj, model_path)   # picklable because no lambdas
print(f"Saved model: {model_path}")

metrics_path = os.path.join(OUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics: {metrics_path}")

info = {
    "purpose": "Light Hindi classifier (TF-IDF + LinearSVC calibrated)",
    "file_path_used": FILE_PATH,
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "random_state": RANDOM_STATE,
    "model_type": "tfidf_word+char + LinearSVC (calibrated)",
    "training_date": pd.Timestamp.now().isoformat()
}
info_path = os.path.join(OUT_DIR, "info.json")
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2)
print(f"Saved info: {info_path}")

print("Done. To use: obj = joblib.load(model.joblib); probs = obj['pipeline'].predict_proba([text]); label = (probs[:,1] >= obj['threshold']).astype(int)")


Step 5/9 — Building TF-IDF + LinearSVC pipeline...
Fast training: subsample training data for speed.
Step 6/9 — Training model (fast or thorough depending on config)...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\sanyam\AppData\Local\Temp\ipykernel_2988\2946698241.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_tr.groupby('label', group_keys=False).apply(lambda g: g.sample(max(1, int(n * len(g)/len(df_tr))), random_state=RANDOM_STATE)).reset_index(drop=True)
c:\Users\sanyam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\sanyam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly

Best params: {'clf__C': 1.0, 'feats__char__ngram_range': (3, 5), 'feats__word__min_df': 2, 'feats__word__ngram_range': (1, 1), 'select__k': 10000}
Best CV f1: 0.9763
Step 7/9 — Calibrating classifier to obtain probabilities...


c:\Users\sanyam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\sanyam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Calibration done.
Step 8/9 — Tuning decision threshold on validation set (maximize F1)...
Chosen threshold on validation: 0.440  (val F1=0.9759)
Step 9/9 — Evaluating on test set and saving artifacts...
Test Accuracy: 0.9771120834735779
Test F1: 0.9773182121414277
Test ROC-AUC: 0.9975279135046164
Classification report:
               precision    recall  f1-score   support

           0     0.9802    0.9736    0.9769      1477
           1     0.9741    0.9806    0.9773      1494

    accuracy                         0.9771      2971
   macro avg     0.9772    0.9771    0.9771      2971
weighted avg     0.9771    0.9771    0.9771      2971

Confusion matrix:
 [[1438   39]
 [  29 1465]]
Saved model: models\light\hindi_light_model\model.joblib
Saved metrics: models\light\hindi_light_model\metrics.json
Saved info: models\light\hindi_light_model\info.json
Done. To use: obj = joblib.load(model.joblib); probs = obj['pipeline'].predict_proba([text]); label = (probs[:,1] >= obj['threshold']).a